In [ ]:
#Installing Groq to access LLMs via API calls(https://groq.com/)
!pip install groq

In [ ]:
#STAGE1: Generating Relevant Intent & Corresponding Business Event Relevant to PS
STAGE1_PROMPT_TEMPLATE = """
You are designing customer-service intelligence for a contact center focused on KEY BUSINESS EVENTS.

Your goal:
Given a DOMAIN and N_QUERIES, generate a realistic set of CUSTOMER INTENTS
(why people call) and the KEY BUSINESS EVENTS that are operationally significant
and require analysis in contact centers. Focus on HIGH-IMPACT events that affect business outcomes, costs, risks, and customer experience. 
Examples of customer intents include: 
Loyalty Program, Delay Management, Replacement vs Refund, Product Feedback, Product Returns, Delivery Delays, 
Cancellation Policies, Upgrade Requests, Technical Support, and Service Complaints. 
This list is not exhaustive — generate realistic, domain-relevant intents beyond these.

DOMAIN EXAMPLES (not exhaustive)
- Consumer Finance, Lending & Banking, Healthcare Provider/Patient Services, Fintech/Payroll-as-a-Service, Insurance, Education & Student Support
- Technology/Contact Center Solutions, Healthcare Payer/Member Services, Financial Services/Debt Relief, Logistics/Delivery Services


KEY BUSINESS EVENT TAXONOMY (HIGH-IMPACT & OPERATIONALLY SIGNIFICANT)
CRITICAL BUSINESS EVENTS TO FOCUS ON:
A. CUSTOMER ESCALATION & RETENTION EVENTS:
   - Supervisor escalation, Manager escalation, Legal/compliance escalation, Churn intent signals, Service cancellation, Customer retention success, Churn prevention

B. FINANCIAL & PAYMENT EVENTS:
   - Refund requests/approvals/denials, Payment failures, Chargebacks, Fee disputes, Loan application approvals/denials, Credit limit changes

C. SERVICE DELIVERY & OPERATIONAL EVENTS:
   - Service outages, Delivery failures, Appointment no-shows, Technician dispatch failures, First-call resolution, Service quality complaints

D. COMPLIANCE & SECURITY EVENTS:
   - Fraud suspicion/confirmation, KYC verification failures, Data privacy breaches, Regulatory compliance issues, Document verification failures

E. PRODUCT & FEEDBACK EVENTS:
   - Product improvement suggestions, Feature requests, Bug reports, Customer feedback trends, Upsell/cross-sell opportunities

F. SENTIMENT & EXPERIENCE EVENTS:
   - Customer frustration escalation, Anger management incidents, Trust restoration, Satisfaction improvement, Negative sentiment loops


WHAT YOU MUST GENERATE
1. CUSTOMER INTENTS
   - These are the reasons customers call that lead to KEY BUSINESS EVENTS.
   - Focus on intents that typically result in operationally significant outcomes.
   - For diversity, scale number of intents based on N_QUERIES:
       - If N_QUERIES ≤ 6 → generate 3–5 intents
       - If 7–15 → generate 6–10 intents
       - If >15 → generate 10–18 intents
   - Create domain-specific intents that align with KEY BUSINESS EVENTS

2. KEY BUSINESS EVENTS - OPERATIONALLY SIGNIFICANT:
   - These must be HIGH-IMPACT events that businesses need to analyze causally.
   - Each intent should produce 1–3 KEY business events from the taxonomy above.
   - Focus on events mentioned in the problem statement: escalations, refunds, churn, product improvements.
   - Events should be measurable, trackable, and operationally relevant.

3. REALISM CONSTRAINTS
   - Generate only KEY BUSINESS EVENTS that would trigger analytical interest.
   - Events should be specific enough for causal analysis.
   - Avoid trivial or routine events that don't require root-cause investigation.
   - Ensure events are domain-appropriate and operationally significant.


OUTPUT FORMAT (STRICT JSON)
Return a single JSON object:
{{
  "domain": "{domain}",
  "n_queries": {n_queries},
  "intents": [
    {{
      "intent": "<customer intent leading to key business events>",
      "short_desc": "<1-line why customers call>",
      "business_events": [
        "<KEY business event 1 - operationally significant>",
        "<KEY business event 2 - operationally significant>",
        "<KEY business event 3 - operationally significant>"
      ]
    }}
  ]
}}

RULES:
- Pure JSON only.
- No markdown.
- No commentary outside JSON.
- FOCUS ON KEY BUSINESS EVENTS: escalations, refunds, churn, product improvements, and other operationally significant outcomes.
"""

In [ ]:
#STAGE2: Generating Queries from those relevant business events
STAGE2_PROMPT_TEMPLATE = """
You are an enterprise analytics query generator for call-center conversational data.
Stage-2 task: generate ONE MECE-aligned analytic query FOR ONE BUSINESS EVENT.

You are analyzing conversations that include:
- Utterances and speaker turns (customer vs agent),
- Agent actions and workflow steps (transfers, escalations, policy references),
- Policy codes or terms (fees, SLAs, refund rules, compliance scripts),
- Temporal aspects (call timing, duration, repeat contacts),
- ASR artifacts (misrecognitions, repeats),
- Linguistic and sentiment indicators (only when clearly relevant).

IMPORTANT:
- Do NOT default to sentiment in every query.
- Only mention sentiment or emotion when it is directly relevant to the BUSINESS_EVENT.
- Prefer operational, policy, behavioral, temporal, and comparative angles.

Input fields:
- DOMAIN: {domain}
- BUSINESS_EVENT: {business_event}
- MECE_CATEGORY: {category}
- SUBCATEGORY: {subcategory}
- DIFFICULTY: {difficulty}   # EASY | MEDIUM | HARD

MECE CATEGORY BEHAVIOR:
- Diagnostic:
  - Ask "why" or "what factors cause" the BUSINESS_EVENT.
  - Focus on root causes: policies, workflows, agent behaviors, timing, miscommunications.
- Descriptive:
  - Ask "what patterns", "how often", "when", or "where".
  - Focus on distributions, recurrence, temporal trends, or segment-level patterns.
- Comparative:
  - Ask "which", "who", or "how does X differ from Y".
  - Compare agents, customer segments, channels, time periods, or policy types.

AVAILABLE SUBCATEGORIES FOR EACH MECE CATEGORY:

DIAGNOSTIC Subcategories:
- Causal Root-Analysis: Identify fundamental causes and root drivers
- Behavioral Cause Analysis: Agent or customer behavior drivers
- Emotional & Linguistic Causality: ONLY when emotion is clearly relevant to business event
- Process / Policy Cause Analysis: Workflow, policy, or procedure issues
- ASR / Misunderstanding Causality: Speech recognition or comprehension failures
- Temporal Causality & Early Signals: Timing-related causes or early indicators
- Counterfactual Causality: "What if" scenarios and alternative outcomes

DESCRIPTIVE Subcategories:
- Pattern Identification: Discover recurring patterns or sequences
- Correlation Analysis: Relationships between variables or events
- Statistical Summarization: Aggregate metrics and distributions
- Temporal Trend Analysis: Time-based patterns and evolution
- Segmented Behavior Analysis: Patterns within specific segments
- Event-Type Pattern Discovery: Relationships between different event types

COMPARATIVE Subcategories:
- Event-Type Comparison: Compare different types of events
- Agent Performance Comparison: Agent-to-agent performance analysis
- Customer Segment Comparison: Different customer group behaviors
- Process / Policy Comparison: Alternative processes or policies
- Cross-Domain Comparison: Cross-domain patterns (use cautiously)
- Scenario-Based Contrastive Analysis: Different scenario outcomes


SUBCATEGORY SELECTION RULES:
- If SUBCATEGORY is a SPECIFIC subcategory (not "AUTO"): Use that exact subcategory as your focus
- If SUBCATEGORY is "AUTO": Choose the MOST APPROPRIATE subcategory from the available options above that best fits the BUSINESS_EVENT

DIFFICULTY:
- EASY:
  - Single-factor, one-clause question (no multi-hop, no counterfactuals).
- MEDIUM:
  - Include 2 drivers or dimensions (e.g., time + agent behavior).
  - Temporal references or segmenting allowed.
- HARD:
  - Multi-hop reasoning, counterfactuals, cross-event or cross-cohort comparisons,
  - May involve "what if" or "under which conditions would" style questions.

OUTPUT FORMAT:
Output exactly ONE JSON object (NOT an array) with fields:
{{
  "domain": "{domain}",
  "business_event": "{business_event}",
  "mece_category": "<Diagnostic|Descriptive|Comparative>",
  "subcategory": "<one valid subcategory for this category>",
  "query": "<analytical query in natural language>",
  "difficulty": "{difficulty}"
}}

The "query" must:
- Explicitly analyze the BUSINESS_EVENT in the DOMAIN.
- Respect MECE_CATEGORY behavior.
- Align with the chosen SUBCATEGORY focus.
- Avoid over-focusing on sentiment; only mention sentiment if clearly justified by BUSINESS_EVENT.

Now generate the JSON object for the given inputs. Output JSON only, no commentary.
"""

In [ ]:
import os
import json
import time
import random
import re
from typing import List, Dict, Any, Optional
from collections import defaultdict
from tqdm import tqdm

from groq import Groq

#Intializing Groq CONFIGURATIONS
GROQ_API_KEY = "gsk_XXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXXX" #Groq API Key
MODEL = "openai/gpt-oss-safeguard-20b" #LLM Model to use for Query generation
MAX_RETRIES = 5 #Retry if Token-Limit is hit(as we were using Free-Tier)
BASE_BACKOFF = 1.0 #For Implementing Exponential Backoff if Token Limit is hit
OUTPUT_DIR = "query_generation_outputs" #Output Directory of saved queries files
os.makedirs(OUTPUT_DIR, exist_ok=True)

# MECE DEFINITIONS (Categories of Types of Queries Possible for Task-1)
MECE_CATEGORIES = ["Diagnostic", "Descriptive", "Comparative"]

# MECE SUB-CATEGORY (Sub-Categories of Types of Queries Possible for Task-1; Read Readme.md and Documentation to know more)
SUBCATEGORY_OPTIONS = {
    "Diagnostic": [
        "Causal Root-Analysis", "Behavioral Cause Analysis", "Emotional & Linguistic Causality",
        "Process / Policy Cause Analysis", "ASR / Misunderstanding Causality",
        "Temporal Causality & Early Signals", "Counterfactual Causality"
    ],
    "Descriptive": [
        "Pattern Identification", "Correlation Analysis", "Statistical Summarization",
        "Temporal Trend Analysis", "Segmented Behavior Analysis", "Event-Type Pattern Discovery"
    ],
    "Comparative": [
        "Event-Type Comparison", "Agent Performance Comparison", "Customer Segment Comparison",
        "Process / Policy Comparison", "Cross-Domain Comparison", "Scenario-Based Contrastive Analysis"
    ]
}

#Ensuring the we get balanced subcategory if subcategory is not explicitly asked for
def get_balanced_subcategory(mece_category: str, category_query_count: int) -> str:
    """Get balanced subcategory distribution using per-category counter"""
    valid_subcategories = SUBCATEGORY_OPTIONS.get(mece_category, [])

    if not valid_subcategories:
        # Fallback to first available subcategory from any category
        for cat in SUBCATEGORY_OPTIONS.values():
            if cat:
                return cat[0]
        return "Causal Root-Analysis"  # Ultimate fallback

    # Use the count of queries in THIS category for round-robin distribution
    subcat_index = category_query_count % len(valid_subcategories)
    return valid_subcategories[subcat_index]

# Initializing GROQ CLIENT
def get_client():
    return Groq(api_key=GROQ_API_KEY)

def backoff_sleep(attempt: int):
    time.sleep(BASE_BACKOFF * (2 ** attempt) + random.random() * 0.2)

# JSON PARSING For Outputs from the LLMs
def parse_json(raw: str) -> Any:
    """Robust JSON parsing with markdown code block handling."""
    if raw is None:
        raise ValueError("No raw text to parse")

    raw = raw.strip()

    # Handle markdown code blocks explicitly
    if raw.startswith('```'):
        # Extract content between code blocks
        lines = raw.split('\n')
        json_lines = []
        in_code_block = False

        for line in lines:
            if line.strip().startswith('```'):
                if in_code_block:
                    break  # End of code block
                else:
                    in_code_block = True
                    continue
            if in_code_block:
                json_lines.append(line)

        if json_lines:
            raw = '\n'.join(json_lines).strip()
        else:
            # Fallback: try to extract any JSON-like content
            raw = re.sub(r'^```(?:json)?\s*', '', raw)
            raw = re.sub(r'\s*```$', '', raw)

    # Additional cleanup for common prefixes
    raw = re.sub(r'^JSON:\s*', '', raw, flags=re.IGNORECASE)
    raw = re.sub(r'^Here(?:\\u2019s|\'s)?\s*(?:the)?\s*(?:JSON|json)?\s*:?\s*', '', raw, flags=re.IGNORECASE)

    # Try direct parse first
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass

    # Find the first complete JSON object with balanced braces
    brace_count = 0
    start_index = -1
    end_index = -1

    for i, char in enumerate(raw):
        if char == '{':
            brace_count += 1
            if start_index == -1:
                start_index = i
        elif char == '}':
            brace_count -= 1
            if brace_count == 0 and start_index != -1:
                end_index = i
                break

    if start_index != -1 and end_index != -1:
        json_str = raw[start_index:end_index + 1]

        # Apply comprehensive cleaning
        json_str = json_str.replace("'", '"')
        json_str = re.sub(r',\s*([}\]])', r'\1', json_str)  # Remove trailing commas
        json_str = re.sub(r'(\w+)\s*:', r'"\1":', json_str)  # Quote unquoted keys

        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            pass

    # Last resort: try to fix common JSON issues more aggressively
    try:
        # Remove all markdown artifacts
        cleaned = re.sub(r'```.*?```', '', raw, flags=re.DOTALL)
        cleaned = re.sub(r'`[^`]*`', '', cleaned)
        cleaned = re.sub(r'\*\*.*?\*\*', '', cleaned)  # Remove bold
        cleaned = re.sub(r'\*.*?\*', '', cleaned)      # Remove italics

        # Find any object-like structure
        match = re.search(r'\{[^{}]*\{[^{}]*\}[^{}]*\}|\{[^{}]*\}', cleaned, re.DOTALL)
        if match:
            candidate = match.group(0)
            candidate = candidate.replace("'", '"')
            candidate = re.sub(r',\s*([}\]])', r'\1', candidate)
            return json.loads(candidate)
    except Exception:
        pass

    # If we get here, try one more approach - manual reconstruction
    raise ValueError(f"Failed to parse JSON from LLM output after multiple attempts")

# FALLBACK QUERY GENERATION(Very Worst Case-Scenario)
def create_fallback_query(domain: str, business_event: str, mece_category: str, subcategory: str, difficulty: str) -> Dict[str, Any]:
    """Create a reliable fallback query when JSON parsing fails."""
    # Simple query templates by category
    query_templates = {
        "Diagnostic": "What are the root causes of {event} in {domain} conversations?",
        "Descriptive": "What patterns are associated with {event} in {domain} conversations?",
        "Comparative": "How does {event} differ across various segments in {domain} conversations?",
    }

    template = query_templates.get(mece_category, "What factors influence {event} in {domain} conversations?")

    return {
        "domain": domain,
        "business_event": business_event,
        "mece_category": mece_category,
        "subcategory": subcategory if subcategory != "AUTO" else get_balanced_subcategory(mece_category, 0),
        "query": template.format(event=business_event, domain=domain),
        "difficulty": difficulty.upper()
    }

#API CALL FUNCTIONS
def call_model_get_text(prompt: str, temp: float = 0.3) -> str:
    """Call Groq API with strict JSON formatting."""
    client = get_client()
    last_err = None

    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=MODEL,
                messages=[
                    {
                        "role": "system",
                        "content": "You are a JSON generator. Always respond with valid JSON only, no markdown, no additional text."
                    },
                    {
                        "role": "user",
                        "content": prompt
                    }
                ],
                temperature=temp,
                response_format={"type": "json_object"}
            )
            content = resp.choices[0].message.content
            if content is None:
                raise ValueError("Empty response from LLM")
            return content.strip()
        except Exception as e:
            last_err = e
            if attempt == MAX_RETRIES - 1:
                raise
            backoff_sleep(attempt)
    raise RuntimeError(f"Model call failed: {last_err}")

def call_groq_chat(client: Groq, prompt: str, model: str = MODEL) -> str:
    """Call Groq API with retry logic for Stage 1."""
    last_err = None
    for attempt in range(MAX_RETRIES):
        try:
            resp = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.8,
                response_format={"type": "json_object"}
            )
            content = resp.choices[0].message.content
            if content is None:
                raise ValueError("Empty response from LLM")
            return content.strip()
        except Exception as e:
            last_err = e
            if attempt == MAX_RETRIES - 1:
                raise
            backoff_sleep(attempt)
    raise RuntimeError(f"Groq failed after retries: {last_err}")

def extract_first_json_blob(text: str) -> Any:
    """Simpler JSON extraction from LLM response."""
    if not isinstance(text, str):
        raise ValueError("Assistant returned non-string content")

    # Strip markdown fences if any
    text = re.sub(r"```(?:json)?", "", text).strip()

    # 1) Try direct JSON
    try:
        return json.loads(text)
    except Exception:
        pass

    # 2) Find first {...} block
    start = text.find("{")
    end = text.rfind("}")
    if start != -1 and end != -1 and end > start:
        candidate = text[start:end+1]
        candidate = candidate.replace("\r", "")
        candidate = candidate.replace("'", '"')
        candidate = re.sub(r",\s*([}\]])", r"\1", candidate)
        try:
            return json.loads(candidate)
        except Exception:
            pass

    raise ValueError("No valid JSON found in response")

def run_stage1(domain: str, n_queries: int) -> Dict[str, Any]:
    """Stage 1: Generate intents and business events from domain."""
    client = get_client()
    prompt = STAGE1_PROMPT_TEMPLATE.format(domain=domain, n_queries=n_queries)

    print(f"Stage 1: Generating business events for '{domain}'...")
    raw = call_groq_chat(client, prompt)
    parsed = extract_first_json_blob(raw)

    if not isinstance(parsed, dict) or "intents" not in parsed:
        raise ValueError("Invalid Stage 1 output structure")

    print(f"Generated {len(parsed['intents'])} intents with business events")
    return parsed

def extract_business_events(stage1_output: Dict[str, Any]) -> List[str]:
    """Extract unique business events from Stage 1 output."""
    business_events = []
    for intent in stage1_output.get("intents", []):
        events = intent.get("business_events", [])
        business_events.extend(events)

    seen = set()
    unique_events = []
    for event in business_events:
        if event not in seen:
            seen.add(event)
            unique_events.append(event)

    return unique_events

def generate_single_query(
    domain: str,
    business_event: str,
    mece_category: str,
    subcategory: str,
    difficulty: str
) -> Dict[str, Any]:
    """Generate one query with enhanced markdown and JSON handling."""
    prompt = STAGE2_PROMPT_TEMPLATE.format(
        domain=domain,
        business_event=business_event,
        category=mece_category,
        subcategory=subcategory,
        difficulty=difficulty.upper()
    )

    # Add explicit JSON formatting instruction
    json_instruction = "\n\nIMPORTANT: Respond with ONLY valid JSON, no markdown code blocks, no additional text."
    full_prompt = prompt + json_instruction

    max_retries = 3
    for retry in range(max_retries):
        try:
            raw = call_model_get_text(full_prompt, temp=0.3)
            parsed = parse_json(raw)

            # Validate the parsed JSON has required fields
            if not isinstance(parsed, dict):
                raise ValueError("Parsed response is not a dictionary")

            required_fields = ["query", "mece_category", "subcategory"]
            missing_fields = [field for field in required_fields if field not in parsed]
            if missing_fields:
                raise ValueError(f"Missing required fields: {missing_fields}")

            break

        except Exception as e:
            print(f"Attempt {retry + 1}/{max_retries} failed: {e}")
            if retry == max_retries - 1:
                print(f"All retries exhausted for event: {business_event}")
                return create_fallback_query(domain, business_event, mece_category, subcategory, difficulty)
            else:
                # Increase temperature slightly on retry
                time.sleep(1)
                continue

    # Ensure all fields are properly set
    parsed.setdefault("domain", domain)
    parsed.setdefault("business_event", business_event)
    parsed.setdefault("mece_category", mece_category)
    parsed.setdefault("difficulty", difficulty.upper())

    # Only override subcategory if it's not properly set
    if "subcategory" not in parsed or not parsed["subcategory"]:
        parsed["subcategory"] = subcategory if subcategory != "AUTO" else get_balanced_subcategory(mece_category, 0)

    return parsed

#MAIN GENERATION FUNCTION
def generate_analytical_queries(
    domain: str,
    difficulty: str,
    n_queries: int = 20,
    mece_category: Optional[str] = None,
    subcategory: Optional[str] = None,
    save_output: bool = True
) -> Dict[str, Any]:
    """
    DEBUGGED MAIN USER FUNCTION: Generate analytical queries with proper subcategory balancing.
    """
    difficulty = difficulty.upper()
    if difficulty not in ["EASY", "MEDIUM", "HARD"]:
        raise ValueError("Difficulty must be EASY, MEDIUM, or HARD")

    print(f"GENERATING ANALYTICAL QUERIES")
    print(f"   Domain        : {domain}")
    print(f"   Difficulty    : {difficulty}")
    print(f"   N Queries     : {n_queries}")
    print(f"   MECE Category : {mece_category or 'Auto-distribution'}")
    print(f"   Subcategory   : {subcategory or 'Auto-selection'}")

    #STAGE 1
    print("\n[STAGE 1] Generating business events...")
    stage1_output = run_stage1(domain, n_queries)
    business_events = extract_business_events(stage1_output)
    if not business_events:
        raise ValueError("No business events extracted from Stage 1.")

    print(f"Extracted {len(business_events)} unique business events.")

    #STAGE 2
    print("\n[STAGE 2] Generating analytical queries...")
    queries = []

    # FIXED: Use per-category counters for proper subcategory balancing
    category_counters = defaultdict(int)

    for i in tqdm(range(n_queries), desc="Generating queries"):
        # Round-robin business events
        event = business_events[i % len(business_events)]

        # Determine MECE category and subcategory with PROPER BALANCING
        if mece_category:
            cat = mece_category
            if subcategory:
                subcat = subcategory  # Use specified subcategory
            else:
                # Use per-category counter for balanced subcategory distribution
                subcat = get_balanced_subcategory(cat, category_counters[cat])
                category_counters[cat] += 1
        else:
            # Auto-distribute across all MECE categories
            cat = MECE_CATEGORIES[i % len(MECE_CATEGORIES)]
            # Use per-category counter for balanced subcategory distribution
            subcat = get_balanced_subcategory(cat, category_counters[cat])
            category_counters[cat] += 1

        try:
            query_obj = generate_single_query(
                domain=domain,
                business_event=event,
                mece_category=cat,
                subcategory=subcat,
                difficulty=difficulty
            )
            queries.append(query_obj)
        except Exception as e:
            print(f"Error generating query {i+1}: {e}")
            # Use enhanced fallback
            fallback = create_fallback_query(domain, event, cat, subcat, difficulty)
            queries.append(fallback)
            print(f"Used fallback query for: {event}")

    #SAVE RESULTS
    results = {
        "metadata": {
            "domain": domain,
            "difficulty": difficulty,
            "n_queries": n_queries,
            "mece_category_filter": mece_category,
            "subcategory_filter": subcategory,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "stage1_business_events": business_events,
        "stage1_intents": stage1_output.get("intents", []),
        "analytical_queries": queries
    }

    if save_output:
        safe_domain = re.sub(r'[^A-Za-z0-9_-]+', '_', domain)
        filename = f"queries_{safe_domain}_{difficulty}_{n_queries}.json"
        filepath = os.path.join(OUTPUT_DIR, filename)

        with open(filepath, "w", encoding="utf-8") as f:
            json.dump(results, f, indent=2, ensure_ascii=False)

        print(f"\nResults saved to: {filepath}")

    # Enhanced Summary
    print("\n GENERATION COMPLETE")
    print(f"   - Business Events    : {len(business_events)}")
    print(f"   - Analytical Queries : {len(queries)}")

    # Category distribution
    category_counts = defaultdict(int)
    subcategory_counts = defaultdict(lambda: defaultdict(int))

    for query in queries:
        cat = query["mece_category"]
        subcat = query["subcategory"]
        category_counts[cat] += 1
        subcategory_counts[cat][subcat] += 1

    print("   - MECE Category Distribution:")
    for cat, count in category_counts.items():
        print(f"     {cat}: {count} queries")

        # Show subcategory breakdown for each category
        if subcategory_counts[cat]:
            for subcat, subcount in subcategory_counts[cat].items():
                print(f"       - {subcat}: {subcount}")

    return results

#TEST FUNCTION
def test_json_parsing():
    """Test the JSON parser with problematic responses."""
    test_cases = [
        #problematic case
        '''```
{
  "domain": "Telecom",
  "business_event": "First-call resolution",
  "mece_category": "Comparative",
  "subcategory": "Process / Policy Comparison",
  "query": "Which policy type is more effective for first-call resolution in telecom customer service?"
}```''',
        # Other common problematic formats
        '''JSON: {
  "domain": "Test",
  "business_event": "Test event",
  "mece_category": "Diagnostic",
  "subcategory": "Causal Root-Analysis",
  "query": "Test query"
}''',
    ]

    for i, test_case in enumerate(test_cases):
        try:
            result = parse_json(test_case)
            print(f"Test {i+1} passed: {result.get('query', 'No query')[:50]}...")
        except Exception as e:
            print(f"Test {i+1} failed: {e}")

# MAIN EXECUTION
if __name__ == "__main__":
    print("QUERY GENERATION SYSTEM")

    # Test JSON parsing first
    print("Testing JSON parsing...")
    test_json_parsing()
    print()

# TEST
print("TESTING THE SUB-CATEGORY BALANCING...")

# Example use-case with Hotel domain
results = generate_analytical_queries(
    domain="Hotel",
    difficulty="Medium",
    n_queries=20,
)


In [ ]:
#TASK 2 COMPONENTS(MECE-CATEGORY: Contextual/Interactive Queries)
FOLLOW_UP_PATTERNS = {
    "Evidence Retrieval": {
        "pattern": "Based on the analysis of {original_query}, what specific evidence supports these findings for {business_event}?",
        "focus": "evidence validation and verification"
    },
    "Contextual Follow-Up": {
        "pattern": "Given the results from '{original_query}', what additional business context should we consider for {business_event}?",
        "focus": "broader business context integration"
    },
    "Clarification / Drill-Down": {
        "pattern": "Following up on '{original_query}', can you clarify the specific circumstances around {business_event} that led to these patterns?",
        "focus": "detailed clarification and precision"
    },
    "Refinement Queries": {
        "pattern": "Building on the analysis from '{original_query}', how would you refine this approach specifically for different scenarios of {business_event}?",
        "focus": "analysis refinement and scope adjustment"
    },
    "Counterfactual / Exception Follow-Up": {
        "pattern": "Based on the findings from '{original_query}', what alternative scenarios or exceptions might change the outcomes for {business_event}?",
        "focus": "alternative scenarios and exception analysis"
    },
    "Multi-Turn Analytical Query Chains": {
        "pattern": "Starting from the analysis of '{original_query}', now explore the deeper implications for {business_event} and suggest actionable improvements.",
        "focus": "sequential analytical reasoning"
    },
    "Cross-Case Evidence Linking": {
        "pattern": "Extending the analysis from '{original_query}', how do these patterns for {business_event} connect to similar cases in our system?",
        "focus": "cross-case pattern recognition"
    }
}

In [ ]:
STAGE3_PROMPT_TEMPLATE = """
You are generating CONTEXTUAL FOLLOW-UP QUERIES that build upon TASK 1 analytical findings.

OBJECTIVE: Create follow-up queries that test the system's ability to:
- Maintain context from previous analytical conversations
- Build upon initial findings with deeper investigation
- Connect related analytical threads
- Demonstrate progressive analytical reasoning

CONVERSATION CONTEXT:
- Domain: {domain}
- Business Event: {business_event}
- TASK 1 Query: "{original_query}"
- TASK 1 Analysis Type: {original_mece_category} - {original_subcategory}
- Follow-up Type: {follow_up_type}

FOLLOW-UP FOCUS: {follow_up_focus}

GENERATION RULES:
1. Create a NATURAL FOLLOW-UP that references the TASK 1 query explicitly
2. The query should build upon the analytical direction started in TASK 1
3. It should require understanding of both the business context AND the previous analysis
4. Make it feel like the next logical question in an analytical conversation
5. Ensure it tests contextual continuity and progressive reasoning

CONTEXT DEPENDENCY LEVELS:
- Low: Only needs basic understanding of previous query
- Medium: Requires specific analytical context from previous findings
- High: Depends heavily on specific results or patterns from TASK 1

DIFFICULTY GUIDELINES:
- EASY: Simple extension or clarification of TASK 1
- MEDIUM: Builds on TASK 1 with 1-2 additional analytical dimensions
- HARD: Complex multi-hop reasoning that transforms TASK 1 findings

OUTPUT FORMAT (JSON):
{{
  "task": "Task 2 - Contextual Follow-up",
  "domain": "{domain}",
  "business_event": "{business_event}",
  "task1_query": "{original_query}",
  "task1_analysis_type": "{original_mece_category} - {original_subcategory}",
  "follow_up_type": "{follow_up_type}",
  "follow_up_query": "<the contextual follow-up query>",
  "difficulty": "{difficulty}",
  "context_dependency": "low/medium/high",
  "analytical_progression": "extension/deepening/transformation"
}}

Generate exactly ONE contextual follow-up query.
"""

In [ ]:
def generate_contextual_followup(
    task1_query_obj: Dict[str, Any],
    follow_up_type: str,
    difficulty: str
) -> Dict[str, Any]:
    """Generate one Task 2 follow-up query based on Task 1 query"""

    follow_up_info = FOLLOW_UP_PATTERNS.get(follow_up_type, {
        "pattern": "Based on '{original_query}', what additional insights can we gain about {business_event}?",
        "focus": "general follow-up analysis"
    })

    prompt = STAGE3_PROMPT_TEMPLATE.format(
        domain=task1_query_obj["domain"],
        business_event=task1_query_obj["business_event"],
        original_query=task1_query_obj["query"],
        original_mece_category=task1_query_obj["mece_category"],
        original_subcategory=task1_query_obj["subcategory"],
        follow_up_type=follow_up_type,
        follow_up_focus=follow_up_info["focus"],
        difficulty=difficulty.upper()
    )

    try:
        raw = call_model_get_text(prompt, temp=0.3)
        parsed = parse_json(raw)

        # Validate and enhance
        parsed.setdefault("task", "Task 2 - Contextual Follow-up")
        parsed.setdefault("domain", task1_query_obj["domain"])
        parsed.setdefault("business_event", task1_query_obj["business_event"])
        parsed.setdefault("task1_query", task1_query_obj["query"])
        parsed.setdefault("task1_analysis_type", f"{task1_query_obj['mece_category']} - {task1_query_obj['subcategory']}")
        parsed.setdefault("follow_up_type", follow_up_type)
        parsed.setdefault("difficulty", difficulty.upper())
        parsed.setdefault("context_dependency", "medium")
        parsed.setdefault("analytical_progression", "extension")

        return parsed

    except Exception as e:
        print(f" Contextual follow-up generation failed: {e}")
        # Intelligent fallback
        fallback_query = follow_up_info["pattern"].format(
            original_query=task1_query_obj["query"],
            business_event=task1_query_obj["business_event"]
        )

        return {
            "task": "Task 2 - Contextual Follow-up",
            "domain": task1_query_obj["domain"],
            "business_event": task1_query_obj["business_event"],
            "task1_query": task1_query_obj["query"],
            "task1_analysis_type": f"{task1_query_obj['mece_category']} - {task1_query_obj['subcategory']}",
            "follow_up_type": follow_up_type,
            "follow_up_query": fallback_query,
            "difficulty": difficulty.upper(),
            "context_dependency": "medium",
            "analytical_progression": "extension"
        }

In [ ]:
import json
import os
import time
from typing import List, Dict, Any, Optional, Union
from collections import defaultdict
from tqdm import tqdm

#DEBUGGED HELPER FUNCTIONS

def load_task1_queries_from_json(file_path: str) -> List[Dict[str, Any]]:
    """Load Task 1 queries from JSON file"""
    try:
        with open(file_path, 'r', encoding='utf-8') as f:
            data = json.load(f)
        if 'queries' in data:
            return data['queries']
        elif 'analytical_queries' in data:
            return data['analytical_queries']
        elif 'task1_queries' in data:
            return data['task1_queries']
        elif isinstance(data, list):
            return data  # If the file is directly a list
        else:
            raise ValueError("No Task 1 queries found in JSON file")

    except Exception as e:
        print(f" Error loading Task 1 queries: {e}")
        return []

#TASK 2 Query Generation

def generate_task2_simple(
    task1_file: str = "balanced_100.json",# It would be preferred to provide well balanced across domains & mece categories queries
    output_file: str = "task2_queries.json",
    queries_per_task1: int = 3,
    difficulty: str = "MEDIUM"
) -> Dict[str, Any]:
    """
    SIMPLE TASK 2 GENERATION: Generate specified {n} Task_2 queries for each Task_1 query
    """

    print(" SIMPLE TASK 2 GENERATION")
    print("=" * 50)

    # Load Task 1 queries
    task1_queries = load_task1_queries_from_json(task1_file)

    if not task1_queries:
        raise ValueError(f"No Task 1 queries found in {task1_file}")

    print(f"Loaded {len(task1_queries)} Task 1 queries from {task1_file}")

    # Show domain distribution
    domains = set(q["domain"] for q in task1_queries)
    domain_counts = defaultdict(int)
    for q in task1_queries:
        domain_counts[q["domain"]] += 1

    print(f" Domains: {list(domains)}")
    print(f" Domain distribution: {dict(domain_counts)}")

    # All contextual subcategories
    contextual_subcats = list(FOLLOW_UP_PATTERNS.keys())
    print(f" Using {len(contextual_subcats)} contextual subcategories")

    # Generate Task 2 queries
    task2_queries = []

    print(f"\n Generating {queries_per_task1} Task 2 queries per Task 1 query...")
    print(f"   Total expected: {len(task1_queries) * queries_per_task1} Task 2 queries")

    for i, task1_query in enumerate(tqdm(task1_queries, desc="Generating Task 2")):
        for j in range(queries_per_task1):
            # Round-robin through contextual subcategories
            subcat_index = (i * queries_per_task1 + j) % len(contextual_subcats)
            follow_up_type = contextual_subcats[subcat_index]

            try:
                task2_query = generate_contextual_followup(
                    task1_query_obj=task1_query,
                    follow_up_type=follow_up_type,
                    difficulty=difficulty
                )
                task2_queries.append(task2_query)

            except Exception as e:
                print(f" Failed to generate Task 2 query {j+1} for '{task1_query['business_event']}': {e}")


    # Prepare results
    results = {
        "metadata": {
            "task1_source": task1_file,
            "task1_queries_count": len(task1_queries),
            "task2_queries_count": len(task2_queries),
            "queries_per_task1": queries_per_task1,
            "difficulty": difficulty,
            "generated_at": time.strftime("%Y-%m-%d %H:%M:%S")
        },
        "task1_queries": task1_queries,
        "task2_queries": task2_queries
    }

    # Save results
    with open(output_file, 'w', encoding='utf-8') as f:
        json.dump(results, f, indent=2, ensure_ascii=False)

    print(f"Task 2 results saved to: {output_file}")

    # Print summary
    print_task2_summary_simple(results)

    return results

def print_task2_summary_simple(results: Dict):
    """Print simple Task 2 generation summary"""
    metadata = results["metadata"]
    task2_queries = results["task2_queries"]

    print("\n TASK 2 GENERATION COMPLETE!")
    print("=" * 40)
    print(f"   Task 1 Queries: {metadata['task1_queries_count']}")
    print(f"   Task 2 Queries: {metadata['task2_queries_count']}")
    print(f"   Queries per Task 1: {metadata['queries_per_task1']}")
    print(f"   Difficulty: {metadata['difficulty']}")

    # Domain distribution
    domain_counts = defaultdict(int)
    for query in task2_queries:
        domain_counts[query["domain"]] += 1

    print(f"\n TASK 2 DOMAIN DISTRIBUTION:")
    for domain, count in domain_counts.items():
        print(f"   • {domain}: {count}")

    # Contextual subcategory distribution
    subcat_counts = defaultdict(int)
    for query in task2_queries:
        subcat_counts[query["follow_up_type"]] += 1

    print(f"\n CONTEXTUAL SUBCATEGORY DISTRIBUTION:")
    for subcat, count in subcat_counts.items():
        print(f"   • {subcat}: {count}")

    # Show samples
    if task2_queries:
        print(f"\n SAMPLE TASK 2 QUERIES:")
        for i, query in enumerate(task2_queries[:3]):
            print(f"   {i+1}. [{query['domain']}] {query['follow_up_query'][:80]}...")

# QUICK GENERATION FUNCTION(Used For Testing)

def quick_task2_generation():
    """Quick function to generate Task_2 queries from results obtained from Task_1 query generation"""

    print("QUICK TASK_2 GENERATION")
    print("Generating 2 Task_2 queries for each of the Task_1 queries")
    print()

    results = generate_task2_simple(
        task1_file="./query_generation_outputs/queries_Hotel_MEDIUM_20.json", #FROM Task_1 Generation
        output_file="task2_60_queries.json",
        queries_per_task1=2,
        difficulty="MEDIUM"
    )

    return results

#MAIN EXECUTION

if __name__ == "__main__":
    #Test Version
    results = quick_task2_generation()

    print("\n ALL DONE!")

In [ ]:
#More Demonstrations of Task1 and Task2 Query Generation
results = generate_analytical_queries(
    domain="Insurance",
    difficulty="Medium",
    n_queries=2,
    mece_category="Descriptive"
)

In [ ]:
results = generate_analytical_queries(
    domain="Healthcare",
    difficulty="Medium",
    n_queries=4,
    mece_category="Diagnostic",
    subcategory="Emotional & Linguistic Causality"
)

In [ ]:
# Uncomment below given code to download generated queries in .zip format
# !zip -r queries_generated.zip query_generation_outputs/